# Boundary value problem

We will now return to solve the boundary value problem

$$
\frac{d^2u}{dx^2} = -\frac{\pi^2}{4} \left(u + 1\right)
$$

Here, we will have Dirichlet boundary conditions, as before, such that $u(0)=0$ and $u(1)=1$.

## Analytic solution

This is just a harmonic oscillator with $k=\pi/2$ (plus the offset term, +1, which is just an offset that could be eliminated with $v\equiv=u+1$). The solutions look like $u(x)=A\cos(kx)+B\sin(kx)-1$. You can solve for $A$ and $B$ using the boundary conditions, yielding:

$$
u(x) = \cos\left(\frac{\pi}{2}x\right) + 2\sin\left(\frac{\pi}{2}x\right) -1
$$

## Shooting method


The first solution we will investigate here is a "shooting" method. We use a stepping ODE solver (in our case, RK4) to "shoot" the differential equation from one boundary to the other. We will then use a root-finding algorithm to find the value of $u(x=1)$. The algorithm will shoot above and below the root, and converge to the right answer at the boundaries.

Note, however, that the internal values will not necessarily be so accurately represented. 

In [ ]:
%matplotlib widget

## Relaxation method


The next solution we will investigate here is a "relaxation" method. This is going to be very similar to PDE solvers later in the semester. We will start with an initial guess, and use Jacobi's algorithm to iterate until the difference between the computed second derivative stabilizes to a constant value. 

In this case, the internal values ARE accurately represented. 

In [ ]:
import numpy as np
import scipy.integrate
import scipy.optimize
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import display, Image, HTML
from bvp import BVPShoot, BVPRelax

### Set up functions

In [ ]:
def simpleBVPFunc(u):
    return -np.pi**2 * (u+1)*0.25

def simpleBVP(x, state):
    u = state[0] ; uprime = state[1]
    du_dx = uprime
    duprime_dx = simpleBVPFunc(u)
    return [du_dx, duprime_dx]

def analytic(x):
    return -1. + np.cos(np.pi*0.5*x)+2.*np.sin(np.pi*0.5*x)

### Evaluate BVP

In [ ]:
xbounds = np.array([0.,1.])
ubounds = np.array([0.,1.])
uprime = 3
bvpshoot = BVPShoot( simpleBVP, xbounds, ubounds, uprime )
bvpshoot.solve()

In [ ]:
N = 100
xbounds = np.array([0.,1.])
ubounds = np.array([0.,1.])
bvprelax = BVPRelax(simpleBVPFunc, xbounds, ubounds, N)
eps = 1e-6

In [ ]:
niter = 0
while bvprelax.change() > eps:
    niter += 1
    bvprelax.relax()
print(f"{niter} iterations")

## Plot solutions

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
xana = np.linspace(0,1,101)
uana = analytic(xana)
ax.plot(xana,uana, label="analytic")
ax.plot(bvpshoot.xvals, bvpshoot.uvals, label="shoot")
ax.plot(bvprelax.xvals, bvprelax.uvals, label="relax")
plt.xlabel("x")
plt.ylabel("u(x)")
ax.legend()
plt.show()

# Compare the performance

In [ ]:
xbounds = np.array([0.,1.])
ubounds = np.array([0.,1.])
uprime = 3
bvpshoot2 = BVPShoot( simpleBVP, xbounds, ubounds, uprime )


In [ ]:
%%timeit -r 10 -n 100 # cell timeit r: number of runs n: number of loops
bvpshoot2.solve()


In [ ]:
N = 100
xbounds = np.array([0.,1.])
ubounds = np.array([0.,1.])
bvprelax2 = BVPRelax(simpleBVPFunc, xbounds, ubounds, N)
eps = 1e-6

In [ ]:
%%timeit -r 10 -n 100 # cell timeit r: number of runs n: number of loops
while bvprelax2.change() > eps:
    bvprelax2.relax()

## Animate solutions

The relaxation method will only print every 1000th frame (!!!)

In [ ]:
from IPython.display import HTML

class AnimatedBVP:
    """Animate a sequence of trial solutions (shooting attempts or relaxation sweeps).
    Display with: HTML(anim.ani.to_jshtml())"""
    def __init__(self, xplot, uplot, nframes=100, interval=50, trail=False,
                 logspace=False, keep_every=None):
        self.xplot, self.uplot = xplot, uplot
        self.trail, self.keep_every = trail, keep_every
        self.trails = []
        n = len(xplot)
        if logspace:   # relaxation slows down: show early sweeps densely, late ones sparsely
            frames = np.unique(np.geomspace(1, n, nframes).astype(int)) - 1
        else:
            frames = np.unique(np.linspace(0, n - 1, min(n, nframes)).astype(int))

        self.fig, self.ax = plt.subplots(figsize=(8, 6))
        xana = np.linspace(0, 1, 101)
        self.ax.plot(xana, analytic(xana), "k-", lw=2, label="analytic")
        if keep_every:   # legend entry only; the kept lines themselves are unlabeled
            self.ax.plot([], [], "--", color="0.6", lw=1, label=f"every {keep_every} iterations")
        (self.line,) = self.ax.plot([], [], "o-", ms=3, color="C1")
        self.ax.set(xlabel="x", ylabel="u(x)", xlim=(0, 1), ylim=(0, 2))
        self.ax.legend(loc="upper left")
        self.reset()
        # repeat=False makes the HTML player default to "Once", so it stops on the last frame
        self.ani = animation.FuncAnimation(self.fig, self.update, frames=frames,
                                           init_func=self.reset, interval=interval,
                                           repeat=False)

    def reset(self):
        for t in self.trails:
            t.remove()
        self.trails = []
        self.next_keep = self.keep_every
        self.line.set_data([], [])
        return (self.line,)

    def update(self, i):
        if self.trail and len(self.line.get_xdata()):   # faint copy of the previous frame
            self.trails += self.ax.plot(self.line.get_xdata(), self.line.get_ydata(),
                                        color="0.8", lw=1, zorder=0)
        # Permanent dashed snapshots at every multiple of keep_every reached so far
        while self.keep_every and self.next_keep <= i:
            k = self.next_keep
            self.trails += self.ax.plot(self.xplot[k], self.uplot[k], "--", color="0.6", lw=1, zorder=1)
            self.next_keep += self.keep_every
        self.line.set_data(self.xplot[i], self.uplot[i])
        self.line.set_label(f"iteration {i}")
        self.ax.legend(loc="upper left")
        return (self.line,)

In [ ]:
anim1 = AnimatedBVP(bvpshoot.xplot, bvpshoot.uplot, interval=500, trail=True)
plt.close(anim1.fig)
HTML(anim1.ani.to_jshtml())

In [ ]:
anim2 = AnimatedBVP(bvprelax.xplot, bvprelax.uplot, logspace=True, keep_every=1000)
plt.close(anim2.fig)
HTML(anim2.ani.to_jshtml())